## 1. Read JSON from ex02

In [4]:
import pandas as pd  # work with tables
import numpy as np   # random numbers, arrays
import requests      # allowed by task, may be used later

# Display float numbers with 2 decimal places
pd.options.display.float_format = "{:.2f}".format

# Read the JSON file from ex02
# NOTE: change the path if your auto.json is in a different place
auto = pd.read_json("../ex02/data/auto.json")

# Just to check structure
auto.head(), auto.dtypes

(      CarNumber  Refund   Fines    Make  Model
 0  Y163O8161RUS       2 3200.00    Ford  Focus
 1   E432XX77RUS       1 6500.00  Toyota  Camry
 2   7184TT36RUS       1 2100.00    Ford  Focus
 3  X582HE161RUS       2 2000.00    Ford  Focus
 4  92918M178RUS       1 5700.00    Ford  Focus,
 CarNumber     object
 Refund         int64
 Fines        float64
 Make          object
 Model         object
 dtype: object)

## 2. Create a sample and concatenate rows

In [5]:
# Create a sample of 200 rows from the original dataframe
# We sample rows from auto itself, so there are no new (CarNumber, Make, Model) combinations
sample_200 = auto.sample(n=200, replace=True, random_state=21)

# Concatenate the sample with the original dataframe
concat_rows = pd.concat([auto, sample_200], ignore_index=True)

concat_rows.head(), concat_rows.shape

(      CarNumber  Refund   Fines    Make  Model
 0  Y163O8161RUS       2 3200.00    Ford  Focus
 1   E432XX77RUS       1 6500.00  Toyota  Camry
 2   7184TT36RUS       1 2100.00    Ford  Focus
 3  X582HE161RUS       2 2000.00    Ford  Focus
 4  92918M178RUS       1 5700.00    Ford  Focus,
 (925, 5))

## 3. Add Year column and build fines dataframe

In [6]:
# Set random seed for reproducibility
np.random.seed(21)

# Create random years from 1980 to 2019 for each row in concat_rows
years = np.random.randint(1980, 2020, size=len(concat_rows))

# Build a Series for the year column
year_series = pd.Series(years, index=concat_rows.index, name="Year")

# Concatenate with concat_rows to get the fines dataframe
fines = pd.concat([concat_rows, year_series], axis=1)

fines.head(), fines.shape

(      CarNumber  Refund   Fines    Make  Model  Year
 0  Y163O8161RUS       2 3200.00    Ford  Focus  1989
 1   E432XX77RUS       1 6500.00  Toyota  Camry  1995
 2   7184TT36RUS       1 2100.00    Ford  Focus  1984
 3  X582HE161RUS       2 2000.00    Ford  Focus  2015
 4  92918M178RUS       1 5700.00    Ford  Focus  2014,
 (925, 6))

## 4. Create owners dataframe (CarNumber + SURNAME)

In [7]:
# Read surnames from the local datasets folder
# File path based on your structure: ex04/datasets/surname.json
surnames_raw = pd.read_json("datasets/surname.json")

surnames_raw.head(), surnames_raw.columns

(          0       1     2
 0      NAME   COUNT  RANK
 1     ADAMS  427865    42
 2     ALLEN  482607    33
 3   ALVAREZ  233983    92
 4  ANDERSON  784404    15,
 RangeIndex(start=0, stop=3, step=1))

In [8]:
# Choose the correct column with surnames
# Change "name" to the real column name if needed
if "name" in surnames_raw.columns:
    surname_col = "name"
else:
    # fallback: use the first column
    surname_col = surnames_raw.columns[0]

# Clean surnames: convert to string and remove special characters like commas, brackets
surnames_clean = (
    surnames_raw[surname_col]
    .astype(str)
    .str.replace(r"[^A-Za-z\- ]", "", regex=True)
    .str.strip()
)

# Drop empty strings if any
surnames_clean = surnames_clean[surnames_clean != ""]

# Convert to Python list
surnames_list = surnames_clean.tolist()

len(surnames_list), surnames_list[:10]

(101,
 ['NAME',
  'ADAMS',
  'ALLEN',
  'ALVAREZ',
  'ANDERSON',
  'BAILEY',
  'BAKER',
  'BENNETT',
  'BROOKS',
  'BROWN'])

In [9]:
# Get the unique car numbers from the 200-row sample
unique_cars_sample = sample_200["CarNumber"].unique()
n_unique_cars = len(unique_cars_sample)

# Make sure we can sample surnames
np.random.seed(21)

if len(surnames_list) >= n_unique_cars:
    # Sample without replacement if we have enough surnames
    chosen_surnames = np.random.choice(
        surnames_list,
        size=n_unique_cars,
        replace=False
    )
else:
    # Fallback: allow repetitions if surnames_list is shorter
    chosen_surnames = np.random.choice(
        surnames_list,
        size=n_unique_cars,
        replace=True
    )

# Build owners dataframe: CarNumber + SURNAME
owners = pd.DataFrame({
    "CarNumber": unique_cars_sample,
    "SURNAME": chosen_surnames
})

owners.head(), owners.shape

(      CarNumber   SURNAME
 0  Y351O8197RUS     REYES
 1   H917TC36RUS    ROGERS
 2  C589EY154RUS   MORALES
 3   K846YE77RUS  ANDERSON
 4  X4108H125RUS      LONG,
 (155, 2))

## 5. Append 5 rows to fines and modify owners (drop last 20, add 3 new)

In [10]:
# Create 5 new fake rows for fines dataframe
extra_fines = pd.DataFrame([
    {
        "CarNumber": "NEWCAR01RUS",
        "Refund": 1.0,
        "Fines": 5000.0,
        "Make": "Tesla",
        "Model": "Model S",
        "Year": 2015,
    },
    {
        "CarNumber": "NEWCAR02RUS",
        "Refund": 2.0,
        "Fines": 7500.0,
        "Make": "BMW",
        "Model": "X5",
        "Year": 2012,
    },
    {
        "CarNumber": "NEWCAR03RUS",
        "Refund": 1.0,
        "Fines": 3000.0,
        "Make": "Lada",
        "Model": "Granta",
        "Year": 2005,
    },
    {
        "CarNumber": "NEWCAR04RUS",
        "Refund": 2.0,
        "Fines": 9000.0,
        "Make": "Audi",
        "Model": "A6",
        "Year": 2018,
    },
    {
        "CarNumber": "NEWCAR05RUS",
        "Refund": 1.0,
        "Fines": 4200.0,
        "Make": "Toyota",
        "Model": "Corolla",
        "Year": 2010,
    },
])

# Concatenate with fines
fines = pd.concat([fines, extra_fines], ignore_index=True)

fines.tail(7)

,CarNumber,Refund,Fines,Make,Model,Year
923,8437XX154RUS,2.00,800.00,Ford,Focus,2005
924,C410X938RUS,2.00,2200.00,Ford,Focus,1997
925,NEWCAR01RUS,1.00,5000.00,Tesla,Model S,2015
926,NEWCAR02RUS,2.00,7500.00,BMW,X5,2012
927,NEWCAR03RUS,1.00,3000.00,Lada,Granta,2005
928,NEWCAR04RUS,2.00,9000.00,Audi,A6,2018
929,NEWCAR05RUS,1.00,4200.00,Toyota,Corolla,2010


In [11]:
# Drop last 20 rows from owners if length is big enough
if len(owners) > 20:
    owners = owners.iloc[:-20].copy()

# Add 3 new owners with CarNumbers different from the 5 extra in fines
extra_owners = pd.DataFrame([
    {"CarNumber": "OWNER01RUS", "SURNAME": "SMITH"},
    {"CarNumber": "OWNER02RUS", "SURNAME": "JOHNSON"},
    {"CarNumber": "OWNER03RUS", "SURNAME": "BROWN"},
])

owners = pd.concat([owners, extra_owners], ignore_index=True)

owners.tail(5), owners.shape

(        CarNumber   SURNAME
 133  Y964O8197RUS  PHILLIPS
 134  934497178RUS  GONZALEZ
 135    OWNER01RUS     SMITH
 136    OWNER02RUS   JOHNSON
 137    OWNER03RUS     BROWN,
 (138, 2))

## 6. Join fines and owners (inner join on CarNumber)

In [12]:
# Inner join: keep only car numbers that exist in both dataframes
fines_owners = fines.merge(owners, on="CarNumber", how="inner")

fines_owners.head(), fines_owners.shape

(      CarNumber  Refund   Fines    Make  Model  Year SURNAME
 0  Y163O8161RUS    2.00 3200.00    Ford  Focus  1989   REYES
 1   E432XX77RUS    1.00 6500.00  Toyota  Camry  1995     COX
 2  92918M178RUS    1.00 5700.00    Ford  Focus  2014  TORRES
 3  H234YH197RUS    2.00 6000.00    Ford  Focus  1990  FOSTER
 4  E40577152RUS    1.00 8594.59    Ford  Focus  1988    HALL,
 (389, 7))

## 7. Pivot table from fines (sum of fines by Make and Year)

In [13]:
# Create a pivot table: index = Make, columns = Year, values = sum of Fines
pivot_fines = fines.pivot_table(
    index="Make",
    columns="Year",
    values="Fines",
    aggfunc="sum"
)

pivot_fines

Year,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
Make,,,,,,,,,,,,,,,,,,,,,
Audi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9000.00,NaN
BMW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,10500.00,NaN,NaN,8594.59,NaN,NaN,6500.00,NaN
Ford,62394.59,395589.17,140383.76,63100.00,111294.59,189583.76,88994.59,121900.00,95989.17,124100.00,...,120183.76,86689.17,147100.00,145494.59,158894.59,203694.59,118794.59,272200.00,285194.59,101100.00
Lada,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Skoda,12494.59,NaN,6900.00,11594.59,1200.00,10294.59,600.00,26700.00,NaN,91400.00,...,3100.00,500.00,500.00,13794.59,1900.00,46394.59,300.00,NaN,156200.00,9500.00
Tesla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5000.00,NaN,NaN,NaN,NaN
Toyota,18500.00,8594.59,2000.00,7200.00,NaN,NaN,16000.00,8000.00,NaN,26400.00,...,28200.00,8594.59,16094.59,NaN,NaN,NaN,13000.00,10600.00,13000.00,34100.00
Volkswagen,30900.00,4600.00,NaN,11794.59,10300.00,34800.00,22400.00,21600.00,NaN,16000.00,...,9100.00,3600.00,2000.00,9600.00,1300.00,11994.59,2100.00,NaN,NaN,NaN
Volvo,NaN,NaN,6800.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10200.00


## 8. Save fines and owners to CSV in data/ (without index)

In [15]:
# Save fines and owners to CSV in data/ folder without index
fines.to_csv("datasets/fines.csv", index=False)
owners.to_csv("datasets/owners.csv", index=False)